In [0]:
from pyspark.sql import functions as F, DataFrame
from datetime import datetime

# ── Configuração ──────────────────────────────────────────────────────────────
SOURCE_TABLE = "homologacao.salutar_saude.raw_corretores"
TARGET_TABLE = "homologacao.salutar_saude.silver_corretores"

DATE_COLS        = ["data_admissao"]
SENIORIDADE_COLS = ["senioridade"]          # initcap + trim (lição: status 'ATIVO'/'ativo')
TRIM_COLS        = ["corretor_nome", "regiao"]  # trim apenas

run_ts = datetime.now()

print(f"Pipeline  : silver_corretores")
print(f"Iniciado  : {run_ts:%Y-%m-%d %H:%M:%S}")
print(f"Origem    : {SOURCE_TABLE}")
print(f"Destino   : {TARGET_TABLE}")

Pipeline  : silver_corretores
Iniciado  : 2026-07-04 18:32:58
Origem    : homologacao.salutar_saude.raw_corretores
Destino   : homologacao.salutar_saude.silver_corretores


In [0]:
df_raw = spark.table(SOURCE_TABLE)
n_raw = df_raw.count()

print(f"[OK] {n_raw:,} registros lidos de {SOURCE_TABLE}")

[OK] 25 registros lidos de homologacao.salutar_saude.raw_corretores


In [0]:
def parse_date(col_name: str) -> F.Column:
    """Normaliza datas para YYYY-MM-DD.
    Suporta os formatos: YYYY-MM-DD, DD-MM-YYYY, DD/MM/YYYY.
    Usa try_to_date para retornar NULL em vez de lançar exceção.
    """
    return F.date_format(
        F.coalesce(
            F.expr(f"try_to_date(`{col_name}`, 'yyyy-MM-dd')"),
            F.expr(f"try_to_date(`{col_name}`, 'dd-MM-yyyy')"),
            F.expr(f"try_to_date(`{col_name}`, 'dd/MM/yyyy')"),
        ),
        "yyyy-MM-dd",
    ).alias(col_name)


def parse_senioridade(col_name: str) -> F.Column:
    """Normaliza senioridade: remove espaços extras e padroniza capitalização.
    Proativo: cobre variantes de case encontradas em outros campos (ex.: 'ATIVO'/'ativo').
    Exemplos: 'júnior' → 'Júnior' | 'SÊNIOR' → 'Sênior' | 'Pleno ' → 'Pleno'
    """
    return F.initcap(F.trim(F.col(col_name))).alias(col_name)


def parse_trim(col_name: str) -> F.Column:
    """Remove espaços extras de campos de texto livres (nome, região)."""
    return F.trim(F.col(col_name)).alias(col_name)


def transform(df: DataFrame, cols: list) -> DataFrame:
    """Aplica todas as transformações de padronização da camada silver.
    Remove a coluna _rescued_data (artefato da camada RAW) e
    adiciona _updated_at com o timestamp de execução.

    Args:
        df  : DataFrame de origem (camada RAW).
        cols: lista de colunas pré-computada fora da função (evita RPC repetido
              ao acessar df.columns dentro de um loop/função no Spark Connect).
    """
    return df.select(
        *[
            parse_date(c)        if c in DATE_COLS         else
            parse_senioridade(c) if c in SENIORIDADE_COLS  else
            parse_trim(c)        if c in TRIM_COLS         else
            F.col(c)
            for c in cols
        ],
        F.lit(run_ts).cast("timestamp").alias("_updated_at"),  # em select, não withColumn
    )


print("[OK] Funções de transformação definidas.")

[OK] Funções de transformação definidas.


In [0]:
# Computa schema uma única vez fora da função (evita RPC repetido no Spark Connect)
silver_cols = [c for c in df_raw.columns if c != "_rescued_data"]
df_silver = transform(df_raw, silver_cols)

# ── Validação de qualidade ────────────────────────────────────────────────────
DQ_COLS = DATE_COLS + ["senioridade"]

null_counts = (
    df_silver
    .select(*[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in DQ_COLS])
    .collect()[0]
    .asDict()
)
unexpected_senioridade = df_silver.filter(
    F.col("senioridade").isNotNull()
    & ~F.col("senioridade").isin("Júnior", "Pleno", "Sênior")
).count()
n_silver = df_silver.count()

print("── Data Quality ───────────────────────────────────────────────────────────")
print(f"  Total de registros               : {n_silver:,}")
print(f"  Correspondência com RAW          : {'[OK]' if n_silver == n_raw else '[WARN] divergência!'}")
for col_name, nulls in null_counts.items():
    tag = "[WARN]" if nulls > 0 else "[OK]  "
    print(f"  {tag} {col_name}: {nulls:,} nulos")
tag = "[WARN]" if unexpected_senioridade else "[OK]  "
print(f"  {tag} senioridade valores inesperados : {unexpected_senioridade:,}")
print("─" * 65)

assert n_silver == n_raw, f"Contagem divergente: RAW={n_raw}, silver={n_silver}"
assert unexpected_senioridade == 0, f"{unexpected_senioridade} valores inesperados em 'senioridade'"

── Data Quality ───────────────────────────────────────────────────────────
  Total de registros               : 25
  Correspondência com RAW          : [OK]
  [OK]   data_admissao: 0 nulos
  [OK]   senioridade: 0 nulos
  [OK]   senioridade valores inesperados : 0
─────────────────────────────────────────────────────────────────


In [0]:
from delta.tables import DeltaTable

# ── Escrita incremental via MERGE ─────────────────────────────────────────────
# Chave de merge: corretor_id (chave primária da tabela)
MERGE_KEY = "corretor_id"

# Deduplica pela chave de merge antes do MERGE
# (Delta exige cardinalidade 1:1 na fonte — múltiplas linhas com a mesma chave causam erro)
df_to_merge = df_silver.dropDuplicates([MERGE_KEY])

if spark.catalog.tableExists(TARGET_TABLE):
    delta_target = DeltaTable.forName(spark, TARGET_TABLE)

    (
        delta_target.alias("target")
        .merge(
            df_to_merge.alias("source"),
            f"target.{MERGE_KEY} = source.{MERGE_KEY}"
        )
        .whenMatchedUpdateAll()       # atualiza corretor existente se houver mudança
        .whenNotMatchedInsertAll()    # insere novo corretor
        # .whenNotMatchedBySourceDelete()  # descomente para remover corretores deletados da RAW
        .execute()
    )
    print(f"[OK] MERGE concluído      : {TARGET_TABLE}")
else:
    # Primeira execução: cria a tabela (carga inicial)
    df_silver.write.format("delta").saveAsTable(TARGET_TABLE)
    print(f"[OK] Carga inicial        : {TARGET_TABLE}")

n_written = spark.table(TARGET_TABLE).count()
print(f"[OK] Registros na tabela  : {n_written:,}")
print(f"[OK] Fim do pipeline      : {datetime.now():%Y-%m-%d %H:%M:%S}")

[OK] Carga inicial        : homologacao.salutar_saude.silver_corretores
[OK] Registros na tabela  : 25
[OK] Fim do pipeline      : 2026-07-04 18:33:17


In [0]:
%sql
SELECT * FROM homologacao.salutar_saude.silver_corretores

corretor_id,corretor_nome,regiao,senioridade,data_admissao,_updated_at
1,Zeca Almeida,Centro-Oeste,Júnior,2023-07-12,2026-07-04T18:32:58.987Z
2,Vinicius Rodrigues,Centro-Oeste,Pleno,2020-12-26,2026-07-04T18:32:58.987Z
3,Xuxa Silva,Sudeste,Júnior,2022-10-11,2026-07-04T18:32:58.987Z
4,Karina Souza,Sul,Júnior,2020-05-24,2026-07-04T18:32:58.987Z
5,Eduardo Rodrigues,Nordeste,Pleno,2020-01-11,2026-07-04T18:32:58.987Z
6,Carla Almeida,Norte,Júnior,2023-07-13,2026-07-04T18:32:58.987Z
7,Rafael Silva,Sul,Júnior,2019-07-11,2026-07-04T18:32:58.987Z
8,Paulo Santos,Centro-Oeste,Júnior,2023-11-06,2026-07-04T18:32:58.987Z
9,Ana Rodrigues,Centro-Oeste,Pleno,2020-06-27,2026-07-04T18:32:58.987Z
10,Xuxa Nunes,Centro-Oeste,Júnior,2022-09-17,2026-07-04T18:32:58.987Z
